# C5 · HPC, Slurm y Seqtk en Apolo/Apolo-Learning

**Curso:** Bioinformática y Biología Computacional · Universidad EAFIT  
**Duración sugerida:** 3 horas  
**Modalidad:** explicación breve → práctica guiada → reto → evidencia reproducible

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UniversidadEAFIT/compubiol_course/blob/master/notebooks/05_hpc_slurm/05_hpc_slurm_seqtk.ipynb)

## Pregunta guía

Un FASTQ excede la capacidad del computador personal. **¿Cómo solicitar recursos, ejecutar en un nodo de cómputo y demostrar que el resultado es correcto?**

### Objetivos

- distinguir nodo de acceso, nodo de cómputo, partición y almacenamiento;
- conectarse y transferir datos con SSH/SCP/rsync;
- diferenciar ejecución interactiva de `sbatch`;
- usar `squeue`, `sacct`, logs y estados;
- submuestrear FASTQ con Seqtk y semilla fija;
- diagnosticar ruta, memoria y tiempo.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/UniversidadEAFIT/compubiol_course.git"
COLAB_DIR = Path("/content/compubiol_course")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

if IN_COLAB and not COLAB_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_DIR)], check=True)
    os.chdir(COLAB_DIR)

start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del curso. Ejecute el notebook desde el repositorio clonado."
    )
os.chdir(ROOT)
os.environ["COURSE_ROOT"] = str(ROOT)
print(f"Raíz del curso: {ROOT}")

## 1. Arquitectura y etiqueta de uso

El nodo de acceso sirve para editar, transferir, inspeccionar y enviar trabajos; no para ejecutar análisis pesados. Slurm asigna CPU, memoria, tiempo y nodos.

```bash
ssh usuario@host-institucional
sinfo
squeue -u "$USER"
```

Las direcciones, VPN, cuentas, particiones y límites cambian. Use siempre la documentación institucional vigente y no los copie de un notebook antiguo.

## 2. Transferencia verificable

```bash
rsync -avP data/ usuario@host:/ruta/proyecto/data/
sha256sum archivo.fastq.gz
```

`rsync` reanuda transferencias y muestra progreso. Un checksum comprueba identidad de bytes, no calidad biológica.

## 3. Anatomía de un trabajo Slurm

In [ ]:
slurm = ROOT / "scripts/module05/run_seqtk.slurm"
print(slurm.read_text(encoding="utf-8"))

Envío con variables de entorno, sin editar el cuerpo:

```bash
INPUT_FASTQ=/ruta/sample.fastq.gz \
OUTPUT_FASTQ=results/sample.subsample.fastq \
N_READS=100000 \
SEED=2026 \
sbatch --export=ALL scripts/module05/run_seqtk.slurm
```

Monitoreo:

```bash
watch -n 2 'squeue -u "$USER"'
sacct -j JOBID --format=JobID,State,ExitCode,Elapsed,ReqMem,MaxRSS
```

## 4. Diagnóstico basado en evidencia

In [ ]:
from pathlib import Path
for log in sorted((ROOT / "data/module05/logs").glob("*.log")):
    print(f"\n--- {log.name} ---")
    print(log.read_text())

| Evidencia | Diagnóstico | Acción razonable |
|---|---|---|
| `No such file or directory` | ruta o montaje incorrecto | comprobar `pwd`, `ls -l`, ruta absoluta y permisos |
| `OUT_OF_MEMORY`, `oom_kill` | memoria excedida | medir `MaxRSS`, reducir lote o solicitar memoria justificada |
| `TIMEOUT` | límite insuficiente o flujo ineficiente | revisar progreso, optimizar o aumentar tiempo con evidencia |

No resuelva todo solicitando recursos máximos: eso aumenta espera y oculta problemas de diseño.

## 5. Verificación de una salida FASTQ

In [ ]:
from pathlib import Path

def fastq_records(path: Path) -> int:
    lines = sum(1 for _ in path.open(encoding="utf-8"))
    if lines % 4:
        raise ValueError(f"{path} no tiene múltiplo de cuatro líneas")
    return lines // 4

print("Lecturas didácticas:", fastq_records(ROOT / "data/module08/good.fastq"))

### Checkpoint

Un trabajo `COMPLETED` solo significa que terminó con éxito técnico. Verifique que la salida exista, no esté vacía, tenga el número esperado de registros y conserve el formato.

## Reto

Adapte el script al clúster, justifique recursos, envíe una prueba pequeña y use `sacct` para comparar lo solicitado con lo utilizado. Diagnostique uno de los logs simulados.